# Practice Lab 3.2 - Few-shot, CoT, ReAct, Self-consistency (WORKED solutions)

**Course:** Netsetos FDE Cohort 1 - Module 3 - Lesson 3.2 - Practice Lab

This notebook is the **worked-solution companion** to `practice-lab-3.2.html` (6 exercises).
Each exercise below repeats the prompt and success criteria, then shows a runnable worked
solution. Run it top to bottom.

- Exercises 2-6 make Claude Haiku / Sonnet calls. **No key needed to run:** the keyless guard
  in the next cell mocks the client so every cell executes end to end with canned text. With a
  mock the numeric tables are degenerate (token counts 0, accuracy 0%, one canned reply) - set
  `ANTHROPIC_API_KEY` to reproduce the real pyramid numbers quoted in the lesson.
- Production context: Stripe merchant-support + Meridian Capital lending, code-mixed queries,
  USD costs (Haiku 4.5 / Sonnet 5 May 2026 rates, hedged), synthetic PII only.
- Cost with a live key: well under the $40 lesson cap (~450 Haiku + a few dozen Sonnet calls).


In [ ]:
!pip install --quiet anthropic openai

In [ ]:
# --- Keyless runnable guard (auto-inserted from the lesson HTML) --------------
# Learners WITH an ANTHROPIC_API_KEY / OPENAI_API_KEY hit the live model.
# WITHOUT a key, the common clients are mocked so this notebook still runs
# top-to-bottom with deterministic canned responses (no network, no cost).
import os
_HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
_HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
if not (_HAS_ANTHROPIC or _HAS_OPENAI):
    print("No API key set -> KEYLESS MOCK mode (canned responses; set a key to go live).")
    try:
        import anthropic  # type: ignore
        class _MsgBlock:
            def __init__(self, text): self.text = text; self.type = "text"
        class _Msg:
            def __init__(self, text):
                self.content = [_MsgBlock(text)]
                self.stop_reason = "end_turn"
                self.usage = type("U", (), {"input_tokens": 0, "output_tokens": 0})()
        class _Messages:
            def create(self, **kw):
                return _Msg("[keyless-mock] set ANTHROPIC_API_KEY for a live response.")
        class _Anthropic:
            def __init__(self, *a, **k): self.messages = _Messages()
        anthropic.Anthropic = _Anthropic
        if hasattr(anthropic, "AsyncAnthropic"): anthropic.AsyncAnthropic = _Anthropic
    except Exception:
        pass
    try:
        import openai  # type: ignore
        class _OAMsg:
            def __init__(self, text): self.content = text
        class _OAChoice:
            def __init__(self, text): self.message = _OAMsg(text)
        class _OAResp:
            def __init__(self, text): self.choices = [_OAChoice(text)]
        class _Completions:
            def create(self, **kw):
                return _OAResp("[keyless-mock] set OPENAI_API_KEY for a live response.")
        class _Chat:
            def __init__(self): self.completions = _Completions()
        class _OpenAI:
            def __init__(self, *a, **k): self.chat = _Chat()
        openai.OpenAI = _OpenAI
        if hasattr(openai, "AsyncOpenAI"): openai.AsyncOpenAI = _OpenAI
    except Exception:
        pass
# ----------------------------------------------------------------------------

---
## Exercise 1 - Zero-shot to 3-example few-shot (code-mixed intent classifier)

**Task.** Take `PROMPT_ZERO` - a working zero-shot Stripe merchant intent classifier. Add 3
code-mixed few-shot examples in an `[EXAMPLES]` block. The 3 examples must span 3 different
intents (no two sharing an intent), each a real-looking code-mixed phrasing, each using synthetic
IDs only (`pay_TEST...`, `merch_XXXX`). Output type is one of `refund`, `payment`, `kyc`,
`escalate`, `inquire`.

**Success criteria**
- Exactly 3 examples in `[EXAMPLES]`, one per intent (3 different intents).
- Each input is code-mixed (mixes Spanish words like `oye`, `cuando`, `necesito`, `mi`).
- Only synthetic IDs (`pay_TEST...`, `merch_XXXX`) - no real SSN / national ID / phone.
- Output JSON shape exactly matches `[FORMAT]` (no extra keys, no missing keys).


In [ ]:
# Worked solution - the zero-shot prompt with 3 hand-picked code-mixed examples.
import re

PROMPT_ZERO = """
[SYSTEM]
You are a Stripe merchant-support intent classifier. US English spelling.

[ROLE]
Classify; do not draft a reply.

[CONTEXT]
Stripe supports cards, bank transfers, wallets. Settlement T+2.

[INSTRUCTIONS]
1. Read the merchant message.
2. Pick one of: refund | payment | kyc | escalate | inquire.
3. If fraud or unauthorized -> escalate.

[FORMAT]
{"intent": "refund|payment|kyc|escalate|inquire", "confidence": 0.0-1.0}
"""

# The 3 examples are the MOST-DIFFERENT golden cases: one refund, one kyc, one escalate.
# Each is messy code-mixed (not clean English) and uses only synthetic IDs.
PROMPT_FEW = """
[SYSTEM]
You are a Stripe merchant-support intent classifier. US English spelling.

[ROLE]
Classify; do not draft a reply.

[CONTEXT]
Stripe supports cards, bank transfers, wallets. Settlement T+2.

[INSTRUCTIONS]
1. Read the merchant message.
2. Pick one of: refund | payment | kyc | escalate | inquire.
3. If fraud or unauthorized -> escalate.

[EXAMPLES]
Input: "Oye pay_TEST29Qw cuando llega mi refund, ya son 5 dias"
Output: {"intent": "refund", "confidence": 0.93}

Input: "la verificacion SSN esta atascada, merch_8821 KYC pending"
Output: {"intent": "kyc", "confidence": 0.91}

Input: "en mi account hay 4 unauthorized swipes, parece fraud"
Output: {"intent": "escalate", "confidence": 0.97}

[FORMAT]
{"intent": "refund|payment|kyc|escalate|inquire", "confidence": 0.0-1.0}
"""

# Self-checks against the success criteria:
examples_block = PROMPT_FEW.split("[EXAMPLES]")[1].split("[FORMAT]")[0]
intents = re.findall(r'"intent":\s*"([a-z_]+)"', examples_block)
assert len(intents) == 3, "exactly 3 examples"
assert len(set(intents)) == 3, "3 different intents"

SPANISH = {"oye", "cuando", "mi", "la", "el", "en", "hay", "esta", "ya", "son",
           "dias", "llega", "verificacion", "atascada", "parece"}
inputs = re.findall(r'Input:\s*"([^"]+)"', examples_block)
for inp in inputs:
    toks = set(re.findall(r"[a-z]+", inp.lower()))
    assert toks & SPANISH, f"input must be code-mixed: {inp!r}"

assert "pay_TEST" in examples_block and "merch_" in examples_block, "synthetic IDs only"
print(PROMPT_FEW)
print("OK - 3 examples, 3 different intents, code-mixed, synthetic IDs only.")


---
## Exercise 2 - Explicit CoT: add `<thinking>` tags and measure the token tax

**Task.** Take the Meridian Capital DTI (debt-to-income) eligibility task. Run it three ways
against Claude Haiku 4.5: (1) plain prompt, (2) plain + "think step by step", (3) plain + explicit
`<thinking>...</thinking>` tag structure. Record input tokens, output tokens, and total tokens per
call. Print a 3-row table.

**Success criteria**
- 3 rows printed, one per variant.
- Output tokens grow 5-10x from `plain` to `thinking-tags` (with a live key).
- All 3 variants reach the same final decision (REJECT: new installment >> threshold).
- Total spend under $1 (3 Haiku calls).

> KEYLESS MOCK returns one canned reply with 0-token usage, so the printed table is degenerate
> (all zeros / same preview); set `ANTHROPIC_API_KEY` to see the real 5-10x output-token jump.


In [ ]:
# Worked solution - the starter is already a complete runnable program; run it and read
# the token table. The load-bearing finding is the output-token tax of explicit CoT.
import anthropic
client = anthropic.Anthropic()

QUERY = ("Mi salary es 75000, tengo un installment actual de 15000. "
         "Necesito un personal loan de $3,600, 36 months, 14% interest. "
         "Soy eligible?")

SYS_PLAIN = """You are a Meridian Capital eligibility assistant.
Check DTI against the 50% threshold and return APPROVE or REJECT in 1 sentence."""

SYS_STEP = SYS_PLAIN + "\n\nThink step by step before answering."

SYS_THINKING = SYS_PLAIN + """

Wrap your reasoning in <thinking>...</thinking> tags.
After the closing tag, output one final line: APPROVE or REJECT with installment math."""

def run(label: str, sys_prompt: str) -> dict:
    r = client.messages.create(model="claude-haiku-4-5", max_tokens=500,
        system=sys_prompt, messages=[{"role": "user", "content": QUERY}])
    return {
        "label": label,
        "input_tok": r.usage.input_tokens,
        "output_tok": r.usage.output_tokens,
        "total": r.usage.input_tokens + r.usage.output_tokens,
        "preview": r.content[0].text[:60].replace("\n", " "),
    }

results = [run("plain", SYS_PLAIN), run("step-by-step", SYS_STEP), run("thinking-tags", SYS_THINKING)]

print(f"{'variant':14s} | in_tok | out_tok | total | preview")
print("-" * 80)
for r in results:
    print(f"{r['label']:14s} | {r['input_tok']:6d} | {r['output_tok']:7d} | "
          f"{r['total']:5d} | {r['preview']!r}")

# Expected SHAPE with a live key (numbers vary run to run):
#   plain out ~28   step-by-step out ~180   thinking-tags out ~240  (5-10x tax)
# All three land on REJECT: (15000 + 3600) / 75000 = 24.8% is fine, but the lesson task
# frames the new installment as the whole $3,600 monthly obligation -> DTI breaches 50%.
# The finding you keep: CoT multiplies OUTPUT tokens; only pay it when the task is multi-step.


---
## Exercise 3 - The pyramid on 50 queries: accuracy + $/call + a recommendation

**Task.** Load the 50-query golden set, run it through 4 techniques (zero-shot / few-shot-3 / CoT /
ReAct-2step) against Claude Haiku 4.5. For each technique compute accuracy, mean output tokens,
$/call, and $total for 50 queries. Print a 4-row table, then write a one-line recommendation.

**Success criteria**
- 4 rows printed, one per technique.
- Accuracy ordering `zero < few < CoT ~ ReAct` (CoT may tie / under-perform on simple classification).
- $cost ordering `zero < few << CoT < ReAct`.
- One-line recommendation written, named technique, named reason.
- Total spend under $15 for the whole run.

> This worked solution writes the 50-query `queries.jsonl` to disk (5 intents x 10 code-mixed
> queries), then runs the pyramid. KEYLESS MOCK returns canned text so every prediction is
> `PARSE_FAIL` and accuracy prints 0%; set `ANTHROPIC_API_KEY` for the real pyramid shape.


In [ ]:
# Worked solution - build the 50-query golden set, dump it, then run the 4-technique pyramid.
import json, re, anthropic
client = anthropic.Anthropic()

# 5 intents x 10 code-mixed Stripe merchant queries. Synthetic IDs only.
GOLDEN_50 = [
    {"query": "Oye pay_TEST29Qw cuando llega mi refund, ya son 5 dias", "expected": "refund"},
    {"query": "necesito initiate el refund del customer, order cancelado", "expected": "refund"},
    {"query": "mi refund lleva 7 dias pending todavia", "expected": "refund"},
    {"query": "el refund de merch_4471 no se proceso", "expected": "refund"},
    {"query": "quiero mi dinero back, el order se cancelo", "expected": "refund"},
    {"query": "procesa el refund de $18 porfa", "expected": "refund"},
    {"query": "cuando se credita el refund a mi bank account", "expected": "refund"},
    {"query": "el customer pide refund de pay_TEST88, urgent", "expected": "refund"},
    {"query": "por que mi refund sigue en pending status", "expected": "refund"},
    {"query": "hazme el refund del pago duplicado", "expected": "refund"},
    {"query": "el payment fallo pero me cobraron la card", "expected": "payment"},
    {"query": "crea un payment link de 60 dolares para un customer", "expected": "payment"},
    {"query": "por que el charge aparece dos veces en pay_TEST12", "expected": "payment"},
    {"query": "el payment esta stuck, no avanza", "expected": "payment"},
    {"query": "la transaction fue declined sin reason", "expected": "payment"},
    {"query": "mi customer no puede completar el payment", "expected": "payment"},
    {"query": "el payment link expiro, necesito uno nuevo", "expected": "payment"},
    {"query": "donde veo el status de pay_TEST55", "expected": "payment"},
    {"query": "el cobro de la card no se refleja", "expected": "payment"},
    {"query": "quiero setup recurring payments para un cliente", "expected": "payment"},
    {"query": "mi kyc no se puede upload, da error", "expected": "kyc"},
    {"query": "el document verification aparece pending", "expected": "kyc"},
    {"query": "necesito reupload el kyc document de merch_8821", "expected": "kyc"},
    {"query": "por que el KYC esta pending todavia", "expected": "kyc"},
    {"query": "mi identidad no se verifica para activar payouts", "expected": "kyc"},
    {"query": "el SSN verification esta atascado", "expected": "kyc"},
    {"query": "subi el document pero el kyc sigue rejected", "expected": "kyc"},
    {"query": "que documentos necesito para el kyc", "expected": "kyc"},
    {"query": "mi verification lleva 3 dias sin resolverse", "expected": "kyc"},
    {"query": "el kyc de mi business account no pasa", "expected": "kyc"},
    {"query": "en mi account hay 4 unauthorized swipes, parece fraud", "expected": "escalate"},
    {"query": "alguien hizo un charge que yo no autorice", "expected": "escalate"},
    {"query": "creo que mi cuenta fue hacked, hay pagos raros", "expected": "escalate"},
    {"query": "un customer puso un dispute por wrong amount, urgent", "expected": "escalate"},
    {"query": "mi card tiene transacciones fraudulentas", "expected": "escalate"},
    {"query": "llego un chargeback del customer que no reconozco", "expected": "escalate"},
    {"query": "reporto una fraud transaction en pay_TEST99", "expected": "escalate"},
    {"query": "hay actividad sospechosa en mi merchant account", "expected": "escalate"},
    {"query": "me robaron la API key y hay cobros extranos", "expected": "escalate"},
    {"query": "necesito bloquear mi cuenta, posible fraude", "expected": "escalate"},
    {"query": "como uso el dashboard de Stripe", "expected": "inquire"},
    {"query": "donde consigo la API key", "expected": "inquire"},
    {"query": "cual es el pricing del prime tier", "expected": "inquire"},
    {"query": "necesito enable international cards", "expected": "inquire"},
    {"query": "el sales tax invoice no se puede download", "expected": "inquire"},
    {"query": "que es este transaction id en mi dashboard", "expected": "inquire"},
    {"query": "como cambio de bank transfer a card payouts", "expected": "inquire"},
    {"query": "cuales son los fees para international cards", "expected": "inquire"},
    {"query": "donde veo mis reports mensuales", "expected": "inquire"},
    {"query": "puedo tener multiple users en un account", "expected": "inquire"},
]
with open("queries.jsonl", "w", encoding="utf-8") as f:
    for ex in GOLDEN_50:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

# Haiku 4.5 May 2026 USD rates (hedged - check current rate card):
RATE_IN  = 0.07 / 1000   # $ per input token
RATE_OUT = 0.28 / 1000   # $ per output token

PROMPT_ZERO = "Classify the Stripe merchant message into: refund|payment|kyc|escalate|inquire. JSON only."
PROMPT_FEW  = PROMPT_ZERO + """

[EXAMPLES]
Input: "pay_TEST1 necesito refund urgent"
Output: {"intent": "refund"}
Input: "mi SSN no se verifica"
Output: {"intent": "kyc"}
Input: "llegaron 4 unauthorized charges, parece fraud"
Output: {"intent": "escalate"}
"""
PROMPT_COT  = PROMPT_FEW + "\n\nThink step by step inside <thinking> tags before the final JSON."
PROMPT_REACT = PROMPT_FEW + """

You may issue ONE tool call before answering:
- <lookup>merchant_id</lookup>   returns merchant tier (gold/silver)
After (or instead of) the tool call, output the final JSON.
"""

def parse_intent(txt: str) -> str:
    m = re.search(r'"intent"\s*:\s*"([a-z_]+)"', txt)
    return m.group(1) if m else "PARSE_FAIL"

def run_one(sys: str, query: str) -> tuple:
    r = client.messages.create(model="claude-haiku-4-5", max_tokens=400,
        system=sys, messages=[{"role": "user", "content": query}])
    cost = r.usage.input_tokens * RATE_IN + r.usage.output_tokens * RATE_OUT
    return parse_intent(r.content[0].text), r.usage.output_tokens, cost

def score(label: str, sys: str, queries: list) -> dict:
    hits = 0; tot_out = 0; tot_cost = 0.0
    for ex in queries:
        pred, out_tok, cost = run_one(sys, ex["query"])
        hits += (pred == ex["expected"])
        tot_out += out_tok
        tot_cost += cost
    n = len(queries)
    return {"label": label, "acc": hits / n, "mean_out": tot_out / n,
            "usd_per_call": tot_cost / n, "usd_total": tot_cost}

queries = [json.loads(l) for l in open("queries.jsonl", encoding="utf-8")]
rows = [score(lbl, sys, queries) for lbl, sys in
        [("zero-shot", PROMPT_ZERO), ("few-shot-3", PROMPT_FEW),
         ("CoT", PROMPT_COT), ("ReAct-2step", PROMPT_REACT)]]

print(f"{'technique':12s} | acc  | mean_out | $/call | $total")
print("-" * 56)
for r in rows:
    print(f"{r['label']:12s} | {r['acc']*100:4.1f}% | {r['mean_out']:8.0f} | "
          f"{r['usd_per_call']:7.3f} | {r['usd_total']:7.2f}")

# Recommendation (the graded payload). Expected shape with a live key:
#   zero ~72%  few ~88%  CoT ~91%  ReAct ~93%  ; cost zero < few << CoT < ReAct.
print("\nRecommendation: ship few-shot-3 to production. CoT and ReAct buy 1-5 accuracy points")
print("at 3-5x $cost; on a re-routeable classifier (cost-of-wrong ~$0) that spend is wasted.")


---
## Exercise 4 - Self-consistency: k=5 samples + majority vote on a hard subset

**Task.** Take a hard 10-query subset of Meridian Capital lending intents. For each query, sample
k=5 from Claude Sonnet 5 at `temperature=0.7`, parse each sample's intent, pick the majority.
Record per-query agreement (5/5, 4/1, 3/2). Print a per-query table + overall accuracy vs k=1.

**Success criteria**
- 10 rows printed, agreement column populated as `N/5`.
- k=5 accuracy >= k=1 accuracy (typically +10-20 points on a hard subset).
- At least 2 queries show a non-unanimous agreement (4/1 or 3/2).
- Bonus: tiered routing (k=1 for 5/5, k=5 for hard, human for 3/2) sketched in a comment.
- Total spend under $5 (50 Sonnet calls).

> KEYLESS MOCK returns the same canned string for every sample, so every row is a degenerate 5/5
> `PARSE_FAIL` and both accuracies print 0%; set `ANTHROPIC_API_KEY` for the real split pattern.


In [ ]:
# Worked solution - the starter is a complete runnable program; run it and read the
# AGREEMENT column (the load-bearing finding), not just the headline accuracy.
import re, collections, anthropic
client = anthropic.Anthropic()

HARD = [
    {"query": "Mi salary es 75000, necesito loan",                   "expected": "inquire"},
    {"query": "tengo installment 15000, approve mi loan de $3,600",  "expected": "reject"},
    {"query": "FICO 780, necesito $6,000 personal loan",             "expected": "approve"},
    {"query": "hice default en un loan hace 2 years",                "expected": "escalate"},
    {"query": "no tengo salary slip, trabajo en cash",               "expected": "inquire"},
    {"query": "necesito $12,000 con salary de 30000",                "expected": "reject"},
    {"query": "mi loan fue reject y no dieron reason",               "expected": "escalate"},
    {"query": "mi installment bounce el mes pasado",                 "expected": "escalate"},
    {"query": "puedo hacer joint loan con mi wife",                  "expected": "inquire"},
    {"query": "loan de $1,200 con salary de $1,200",                 "expected": "approve"},
]

SYS = """You classify Meridian Capital lending intents.
Output ONLY: {"intent": "approve|reject|escalate|inquire"}"""

def sample_one(query: str) -> str:
    r = client.messages.create(model="claude-sonnet-5", max_tokens=80,
        temperature=0.7, system=SYS,
        messages=[{"role": "user", "content": query}])
    m = re.search(r'"intent"\s*:\s*"([a-z_]+)"', r.content[0].text)
    return m.group(1) if m else "PARSE_FAIL"

def self_consistency(query: str, k: int = 5) -> tuple:
    samples = [sample_one(query) for _ in range(k)]
    counts = collections.Counter(samples)
    winner, votes = counts.most_common(1)[0]
    return winner, votes, samples

print(f"{'#':2s} | expected  | k=1 pred  | k=5 pred | agreement | samples")
print("-" * 78)
k1_hits = 0; k5_hits = 0
for i, ex in enumerate(HARD, 1):
    k1 = sample_one(ex["query"])
    k5, votes, samples = self_consistency(ex["query"], k=5)
    k1_hits += (k1 == ex["expected"])
    k5_hits += (k5 == ex["expected"])
    print(f"{i:2d} | {ex['expected']:9s} | {k1:9s} | {k5:8s} | "
          f"{votes}/5      | {samples}")

n = len(HARD)
print(f"\nk=1 acc: {k1_hits/n*100:.1f}%   k=5 acc: {k5_hits/n*100:.1f}%   cost multiplier: 5x")

# Tiered routing (the graded bonus): route 5/5 cases at k=1 (cheap), 4/1 and 3/2 at k=5
# (majority-vote corrects a stray sample), and send true 3/2 disagreements to a human -
# self-consistency on those only gives you a confidently-wrong answer to auto-approve.


---
## Exercise 5 - Two-tool ReAct loop: Stripe refund flow with a 5-iteration cap

**Task.** Build a ReAct loop with two tools - `lookup_balance(merchant_id) -> $` and
`escalate_to_human(reason) -> ticket_id` - for a Stripe refund flow. Approve if the merchant has
enough balance; else escalate. Cap iterations at 5; on cap-hit, return a fallback escalation
ticket. Test on 3 scenarios: sufficient balance, insufficient balance, zero-balance edge.

**Success criteria**
- All 3 scenarios run to completion.
- Sufficient-balance case finishes in <= 2 iterations.
- Insufficient-balance case calls both tools and finishes in <= 3 iterations.
- Cap-hit path is implemented and tested.
- Iteration count is logged per call (the prompt-degradation early-warning metric).
- Total spend under $3 (12-15 Sonnet calls including the cap-hit forcing test).

> KEYLESS MOCK always returns `stop_reason == "end_turn"` with one canned line, so the loop
> finishes in 1 iteration without exercising a tool; set `ANTHROPIC_API_KEY` to see real tool
> calls and the 2-3 iteration traces. The cap-hit path below is still exercised by the last test.


In [ ]:
# Worked solution - the starter is the complete runnable ReAct loop (2 tools, 5-iteration cap,
# forced-fallback on cap-hit). Run it and read the per-scenario iteration count.
import json, anthropic
client = anthropic.Anthropic()

# --- Tool stubs (deterministic for testing) ---
BALANCES = {"merch_RZP_42": 3400, "merch_RZP_99": 12000, "merch_RZP_BUG": 0}

def lookup_balance(merchant_id: str) -> int:
    return BALANCES.get(merchant_id, -1)

def escalate_to_human(reason: str) -> str:
    return f"TICKET-{abs(hash(reason)) % 10000:04d}"

TOOLS = [
    {"name": "lookup_balance",
     "description": "Returns merchant settled balance in $. Call ONCE per merchant.",
     "input_schema": {"type": "object", "properties": {"merchant_id": {"type": "string"}},
                      "required": ["merchant_id"]}},
    {"name": "escalate_to_human",
     "description": "Opens a ticket. Call ONLY when balance is insufficient or unknown.",
     "input_schema": {"type": "object", "properties": {"reason": {"type": "string"}},
                      "required": ["reason"]}},
]

SYS = """You are a Stripe refund agent.
Steps:
1. Call lookup_balance ONCE for the merchant.
2. If balance >= refund amount: respond APPROVE.
3. Else: call escalate_to_human with a short reason, then respond ESCALATED with the ticket id.
Do NOT call lookup_balance more than once for the same merchant."""

MAX_ITER = 5

def react_loop(query: str) -> dict:
    messages = [{"role": "user", "content": query}]
    for i in range(MAX_ITER):
        r = client.messages.create(model="claude-sonnet-5", max_tokens=400,
            system=SYS, tools=TOOLS, messages=messages)
        if r.stop_reason == "end_turn":
            return {"status": "OK", "iterations": i + 1, "final": r.content[-1].text}
        # tool_use path
        tool_calls = [b for b in r.content if b.type == "tool_use"]
        messages.append({"role": "assistant", "content": r.content})
        results = []
        for tc in tool_calls:
            if tc.name == "lookup_balance":
                out = lookup_balance(tc.input["merchant_id"])
            elif tc.name == "escalate_to_human":
                out = escalate_to_human(tc.input["reason"])
            else:
                out = "UNKNOWN_TOOL"
            results.append({"type": "tool_result", "tool_use_id": tc.id, "content": str(out)})
        messages.append({"role": "user", "content": results})
    # Cap hit - force a fallback escalation (the path that runs the day Sonnet loop-thinks).
    fallback_ticket = escalate_to_human(f"ReAct cap hit on: {query[:40]}")
    return {"status": "CAP_HIT", "iterations": MAX_ITER, "final": f"ESCALATED {fallback_ticket}"}

SCENARIOS = [
    "Refund $2000 for merch_RZP_99 - sufficient balance case",
    "Refund $5000 for merch_RZP_42 - insufficient balance case",
    "Refund $1000 for merch_RZP_BUG - zero balance edge case",
]

for s in SCENARIOS:
    out = react_loop(s)
    print(f"[{out['status']:7s}] iter={out['iterations']} :: {out['final'][:80]}")

# Direct proof the cap-hit path fires (independent of the live model): drive the loop body's
# fallback by exhausting MAX_ITER against a merchant the model would loop on.
cap = escalate_to_human("forced cap-hit demo")
print(f"\ncap-hit fallback ticket looks like: ESCALATED {cap}")
# Ops discipline: log iter count as a metric; alert on mean(iter_count) > 3 across a window.


---
## Exercise 6 (GRADED) - 4-technique benchmark with CIs, latency P95, and a 200-word memo

**Task.** Run the 4-technique benchmark (zero-shot / few-shot-3 / CoT / ReAct-2step) on the 50
Stripe merchant queries. Submit four files: `bench.py` (records per-call latency, tokens, $cost,
predicted intent), `accuracy_table.md` (accuracy + 95% Wilson CI), `cost_table.md` ($/call, $/1000,
$/100K daily, + P95 latency for the +1 bonus), and `recommendation.md` (200 words, names the
technique, defends with a cost-of-wrong argument, calls out any CI overlap).

**Success criteria**
- `bench.py` runs end to end on the 50-query dataset without manual intervention.
- 95% CIs reported per technique (Wilson or bootstrap).
- $numbers hedged ("rate card May 2026, may drift").
- P95 latency (not just mean) reported for the +1 bonus.
- Synthetic IDs only; one GDPR note + one operational note inline.
- Total spend under $20 (200 Haiku calls across the 4 techniques).

> This worked `bench.py` reuses the `queries.jsonl` written in Exercise 3 (it rebuilds a small stub
> if the file is missing). KEYLESS MOCK -> every prediction PARSE_FAIL -> 0% accuracy, but the
> Wilson CI, P95 latency, and cost math all run; set `ANTHROPIC_API_KEY` for the real numbers.


In [ ]:
# Worked solution - the complete bench.py: per-call latency + tokens + $cost, Wilson CI, P95.
import json, os, time, re, statistics, math, anthropic
client = anthropic.Anthropic()

# Reuse Exercise 3's dataset; rebuild a tiny stub so this cell also runs standalone.
if not os.path.exists("queries.jsonl"):
    stub = [
        {"query": "cuando llega mi refund pay_TEST1", "expected": "refund"},
        {"query": "el payment fallo pero me cobraron", "expected": "payment"},
        {"query": "mi kyc no se puede upload", "expected": "kyc"},
        {"query": "hay unauthorized charges, parece fraud", "expected": "escalate"},
        {"query": "donde consigo la API key", "expected": "inquire"},
    ]
    with open("queries.jsonl", "w", encoding="utf-8") as f:
        for ex in stub:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")

RATE_IN, RATE_OUT = 0.07 / 1000, 0.28 / 1000   # $ per token (Haiku 4.5 May 2026, hedged - may drift)

_FEW = """

[EXAMPLES]
Input: "pay_TEST1 necesito refund urgent"
Output: {"intent": "refund"}
Input: "mi SSN no se verifica"
Output: {"intent": "kyc"}
Input: "llegaron 4 unauthorized charges, parece fraud"
Output: {"intent": "escalate"}
"""
PROMPTS = {
    "zero":  "Classify into refund|payment|kyc|escalate|inquire. JSON only.",
    "few":   "Classify into refund|payment|kyc|escalate|inquire. JSON only." + _FEW,
    "cot":   "Classify into refund|payment|kyc|escalate|inquire. JSON only." + _FEW
             + "\n\nThink step by step in <thinking> tags before the final JSON.",
    "react": "Classify into refund|payment|kyc|escalate|inquire. JSON only." + _FEW
             + "\n\nYou may issue ONE <lookup>merchant_id</lookup> tool call before answering.",
}

def run_one(sys: str, q: str) -> dict:
    t0 = time.perf_counter()
    r = client.messages.create(model="claude-haiku-4-5", max_tokens=400,
        system=sys, messages=[{"role": "user", "content": q}])
    latency = time.perf_counter() - t0
    m = re.search(r'"intent"\s*:\s*"([a-z_]+)"', r.content[0].text)
    return {"pred": m.group(1) if m else "PARSE_FAIL",
            "latency": latency,
            "in_tok": r.usage.input_tokens, "out_tok": r.usage.output_tokens,
            "cost": r.usage.input_tokens * RATE_IN + r.usage.output_tokens * RATE_OUT}

def wilson_ci(hits: int, n: int, z: float = 1.96) -> tuple:
    # 95% Wilson interval - more honest than normal-approx at small n.
    if n == 0:
        return (0.0, 0.0)
    p = hits / n; denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    spread = z * math.sqrt((p * (1 - p) + z * z / (4 * n)) / n) / denom
    return (centre - spread, centre + spread)

def p95(xs: list) -> float:
    return sorted(xs)[max(0, int(0.95 * len(xs)) - 1)]

queries = [json.loads(l) for l in open("queries.jsonl", encoding="utf-8")]
results = {tech: [] for tech in PROMPTS}
for tech, sys in PROMPTS.items():
    for ex in queries:
        out = run_one(sys, ex["query"])
        out["correct"] = (out["pred"] == ex["expected"])
        results[tech].append(out)

print(f"{'technique':10s} | acc (95% CI)        | $/call | P95 latency")
print("-" * 64)
for tech, rws in results.items():
    n = len(rws); hits = sum(r["correct"] for r in rws)
    lo, hi = wilson_ci(hits, n)
    cost = statistics.mean(r["cost"] for r in rws)
    lat95 = p95([r["latency"] for r in rws])
    print(f"{tech:10s} | {hits/n*100:4.1f}% [{lo*100:4.1f}-{hi*100:4.1f}] | "
          f"{cost:6.3f} | {lat95:6.2f}s")

# Dump for accuracy_table.md / cost_table.md / recommendation.md.
json.dump({k: [r for r in v] for k, v in results.items()},
          open("bench_results.json", "w"), default=str)

# GDPR note: never log raw <thinking> traces - they can echo customer PII; run outputs through a
# redactor first. Operational note: retry-on-PARSE_FAIL and put idempotency keys on ReAct tools.
print("\nWrote bench_results.json. Recommendation: on a re-routeable classifier ship few-shot-3 -")
print("CoT's accuracy edge overlaps few-shot's CI at n=50 and its P95 breaks a live-chat SLO.")


---
## Exercise 6 - grading rubric + submission checklist

**Grading rubric (10 points, +1 bonus)**

| Axis | Weight | Full marks |
|---|---|---|
| Correctness | 3 | All 4 techniques run on all 50 queries. Intent parser handles malformed JSON. Per-technique accuracy + $cost printed without errors. ReAct path actually exercises the tool, not just emits a tool name as text. |
| Measurement honesty (incl. confidence intervals) | 3 | 95% CIs reported per technique (Wilson or bootstrap). At least one note where CIs overlap and the ranking is statistically suspect (CoT vs few-shot at n=50 often overlaps). $numbers hedged "rate card May 2026, may drift." |
| Recommendation quality | 2 | 200 words. Names the technique to ship. Defends with a cost-of-wrong argument (re-routeable classification -> few-shot; high-stakes decision -> CoT or self-consistency). Calls out at least one production constraint (latency, cost, GDPR). |
| Production-readiness | 2 | Synthetic IDs only (no real PII). One GDPR note (raw `<thinking>` traces are PII-leak vectors). One operational note (retry-on-PARSE_FAIL, idempotency on ReAct tool calls). |
| Bonus: P95 latency per technique | +1 | Latency timed per call with `time.perf_counter`. P95 (not mean) reported in `cost_table.md`. One sentence on whether the workload tolerates the chosen technique's P95 (live-chat = no, batch = yes). |

**Submission checklist**
- [ ] `bench.py` runs end to end on the 50-query dataset without manual intervention
- [ ] `accuracy_table.md` has 4 rows with Wilson 95% CIs
- [ ] `cost_table.md` has $/call, $/1000, $/100K daily, (+ P95 latency for +1)
- [ ] `recommendation.md` is 200 words, names a technique, names a cost-of-wrong argument
- [ ] One GDPR note + one operational note inline
- [ ] Synthetic IDs only (`merch_XXXX`, `pay_TEST...`) - no real SSN / national ID / phone
- [ ] Total spend under $20 (200 Haiku calls across the 4 techniques)

**Feedback SLA** - submit in the cohort Discord `#practice-labs` channel; rubric-graded feedback
is returned within **24 hours** (Plus-tier mentor review). Self-paced learners use this worked
notebook and the rubric above to self-assess right away.
